# Work Partitioning (Preparation for Multi-GPU)

To prepare the stencil solver for multi-GPU execution, the computational domain is partitioned into independent subregions called **patches**.
Each patch describes a rectangular region of the global grid that can later be assigned to a GPU (or be processed in parallel streams on a single device).
For now, this partitioning is only with respect to coordinates - we will retain one big array for the whole domain.

The figure below visualizes this process for one patch (the original configuration), two and four patches.

<div style="text-align: center;">
  <table>
    <tr>
      <td style="text-align: center;">
        <img src="./img/patch-single.png" height="320" alt="Computational Domain"><br>
        <div>Computational domain (single patch).</div>
      </td>
      <td style="text-align: center;">
        <img src="./img/patch-split-2.png" height="320" alt="Computational Domain Split into Two Patches"><br>
        <div>Split into two patches.</div>
      </td>
      <td style="text-align: center;">
        <img src="./img/patch-split-4.png" height="320" alt="Computational Domain Split into Four Patches"><br>
        <div>Split into four patches.</div>
      </td>
    </tr>
  </table>
</div>

Note that we employ a 1D only decomposition.
In practice, 2D decompositions have certain advantages and are usually favored.
They are, however, also a lot harder to implement and optimize for.

### 1. Introduce a Patch Struct

To keep the implementation structured, we introduce a patch abstraction capturing:
* index bounds `[globalInnerBeginX : globalInnerEndX)` x `[globalInnerBeginY : globalInnerEndY)` representing the index space this patch is responsible for, and
* execution configuration for this patch used in kernel launches.

```c++
struct Patch {
    // patch boundaries in global coordinates excluding boundaries
    size_t globalInnerBeginX;
    size_t globalInnerEndX;
    size_t globalInnerBeginY;
    size_t globalInnerEndY;

    // execution configuration
    dim3 blockSize;
    dim3 gridSize;
};
```

Note that the patch indices may be relative to the allocation they work on.
Currently they are **global indices** since there is only one allocation so far, but this will change in later parts of the course.

### 2. Allocating Patch Structs

We can now use the just implemented struct to allocate a number of patches.
For the sake of simplicity, the number of patches will be kept fixed for now.
Alternatively, it could also be adapted dynamically (which we will do later) or be set via the command line.

```c++
constexpr int numPatches = 8; // fixed number of patches

Patch *patches = new Patch[numPatches];
```

Deleting the auxiliary data structure works as usual.

```c++
delete[] patches;
```


### 3. Filling the Patches

After allocating the patches, it is time to initialize them.
For this we first need to calculate the share of the work each patch will be responsible for.
Considering a 1D partitioning only for now, this can be represented by a **number of rows** per patch.
In total, there are `ny - 2` rows that need to be distributed, so a naive approach is simply dividing this number by the number of patches.

```c++
// this only works if there is no remainder
patchHeight = (ny - 2) / numPatches;
```


Below is an illustration for **5** (inner) rows and **2** patches.
As evident, simply dividing leaves untreated rows in cases where the number of rows is not evenly divisible by the number of patches.

<div style="text-align: center;">
  <img src="./img/patch-split-wrong-floor.png" height="320" alt="Untreated Rows in Patches">
</div>

A more robust approach is using a ceiling divide (a divide plus a round up operation).

```c++
patchHeight = ceilingDivide(ny - 2, numPatches);
```


Below is the same illustration as before, but now with the adapted patch height computation.
While we now treat all rows necessary, the last patch also updates extra rows (in this case boundary, in practice this could also be an out-of-bounds access).

<div style="text-align: center;">
  <img src="./img/patch-split-wrong-ceil.png" height="320" alt="Too Many Rows Updated in Patches">
</div>

We can intuitively understand that the height per patch must not be constant.
Instead, we interpret the `patchHeight` as default or **maximum patch height** and adapt each patch's coordinates separately.

```c++
// no partitioning in the x-dimension
patch.globalInnerBeginX = 1;
patch.globalInnerEndX = globalNumCellsX - 1;

patch.globalInnerBeginY = 1 + patchIdx * patchHeight;
patch.globalInnerEndY   = std::min(     // the end is either
    1 + (patchIdx + 1) * patchHeight,   // the beginning of the next patch, or
    globalNumCellsY - 1);               // the end of the global domain
```

The snippet captures all key points:
* There is no partitioning in the x-dimension.
* The first interior cell starts at index `1`.
* Each patch begins where the previous one ends (i.e. there is no overlap between patches).
* The final patch clamps `globalInnerEndY` to `ny - 1` to avoid exceeding the domain (for reasonable large values of `ny`).
* Together, all patches fully cover the interior region `[1, ny - 1)` without overlap or gaps.

### 3.1. Determining Execution Configurations

We use an arbitrary block size of 16 by 16.
Without grid-stride loops, we map each cell of the current patch to a single thread.

```c++
auto numInnerCellsX = patch.globalInnerEndX - patch.globalInnerBeginX;
auto numInnerCellsY = patch.globalInnerEndY - patch.globalInnerBeginY;

patch.blockSize = dim3(16, 16);
patch.gridSize  = dim3(
    ceilingDivide(numInnerCellsX, patch.blockSize.x),
    ceilingDivide(numInnerCellsY, patch.blockSize.y));
```


### 3.2. Wrap a Loop Around

The last step is constructing a loop over all patches in whose body we assign the patch parameters as detailed above.

```c++
for (int patchIdx = 0; patchIdx < numPatches; ++patchIdx) {
    auto &patch = patches[patchIdx];

    // assign patch parameters
}
```


### 4. Launch Patch-Kernels

Now that each patch knows the index space it is responsible for, the next step is adapting the computations.
We start by extending our kernel's parameter list to include the `globalInnerBeginX`, `globalInnerBeginY`, `globalInnerEndX`, and `globalInnerEndY` to be used.

This allows us to map from **thread indices** (equal to patch inner indices) to **global indices**.

```c++
const size_t tidx = blockIdx.x * blockDim.x + threadIdx.x;
const size_t tidy = blockIdx.y * blockDim.y + threadIdx.y;

const size_t i0 = globalInnerBeginX + tidx;
const size_t i1 = globalInnerBeginY + tidy;
```


The condition to check for whether the current thread has valid work now simplifies to a check against the end coordinates.

```c++
if (i0 < globalInnerEndX && i1 < globalInnerEndY) {
    ...
}
```


The final step is looping over all patches and launching one kernel each with the patch's information.

```c++
for (int patchIdx = 0; patchIdx < numPatches; ++patchIdx) {
    const auto &patch = patches[patchIdx];

    stencil2D<<<patch.gridSize, patch.blockSize>>>(
        u, uNew,                                            // global arrays
        patch.globalInnerBeginX, patch.globalInnerEndX,     // global x interval coordinates for the current patch
        patch.globalInnerBeginY, patch.globalInnerEndY,     // global y interval coordinates for the current patch
        globalNumCellsX                                     // global stride required for index linearization
    );
}
```


## Exercise

Choose one of the difficulty levels below to tailor the exercise to your preferences.
Each level provides a different starting point implementation to be copied into [stencil-2d.cu](../src/04-work-partitioning/stencil-2d.cu).
Use it for your implementation and follow the steps outlined above.

Below the difficulty level descriptions, there are cells for compiling, executing and profiling the your solution.

### Level Hard

Create and empty file [stencil-2d.cu](../src/04-work-partitioning/stencil-2d.cu) and copy one of the previous code versions into it (your work or a solution).

In [ ]:
!touch ../src/04-work-partitioning/stencil-2d.cu

### Level Medium

[stencil-2d-medium.cu](../src/04-work-partitioning/stencil-2d-medium.cu) contains a partial solution with TODOs ranging from straight-forward to complex.
Copy the provided code into the working file [stencil-2d.cu](../src/04-work-partitioning/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
%%bash
if [ -e ../src/04-work-partitioning/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/04-work-partitioning/stencil-2d-medium.cu ../src/04-work-partitioning/stencil-2d.cu
fi

### Level Easier

[stencil-2d-easier.cu](../src/04-work-partitioning/stencil-2d-easier.cu) contains a partial solution with TODOs.
The solution is further progressed than the level medium version, and the complexity of the TODOs is limited.
Copy the provided code into the working file [stencil-2d.cu](../src/04-work-partitioning/stencil-2d.cu) with the cell below, then finish the implementation provided there.

In [ ]:
%%bash
if [ -e ../src/04-work-partitioning/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/04-work-partitioning/stencil-2d-easier.cu ../src/04-work-partitioning/stencil-2d.cu
fi

### Possible Solution

[stencil-2d-solution.cu](../src/04-work-partitioning/stencil-2d-solution.cu) contains a possible solution for this exercise.
Copy it into the working file [stencil-2d.cu](../src/04-work-partitioning/stencil-2d.cu) with the cell below.

In [ ]:
%%bash
if [ -e ../src/04-work-partitioning/stencil-2d.cu ]; then
  echo "error: target file already exists"
else
  cp ../src/04-work-partitioning/stencil-2d-solution.cu ../src/04-work-partitioning/stencil-2d.cu
fi

### Compilation, Execution and Profiling

The new code version is available in [04-work-partitioning/stencil-2d.cu](../src/04-work-partitioning/stencil-2d.cu) (after creating it with one of the commands above).
It can be compiled and executed with the following cells.

In [ ]:
!nvcc -O3 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_86,code=sm_86 -o ../build/04-work-partitioning ../src/04-work-partitioning/stencil-2d.cu

The next cell produces output for a small grid.
Visualize the output using the [visualize](./99-visualize.ipynb) notebook after executing the application.

In [ ]:
!../build/04-work-partitioning 256 64 2 2000 100

The next cell produces no output and runs a larger grid.
Use it for performance evaluation.

In [ ]:
!../build/04-work-partitioning $((32 * 1024)) 256 2 16 0

Instead of running on the local **single A40** GPU, we can also submit a batch job running on a **single A100** GPU.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:1 \
    --time 00:05:00 --wait \
    --output=../output/04-work-partitioning.out --error=../output/04-work-partitioning.err \
    --wrap="../build/04-work-partitioning $((32 * 1024)) 256 2 8 0"

cat ../output/04-work-partitioning.out

The next cell performs profiling with Nsight Systems by submitting a batch job.

The profile is then available at [profiles/04-work-partitioning.nsys-rep](../profiles/04-work-partitioning.nsys-rep) and can be downloaded by **shift + right-clicking** the link, by clicking the link with the **middle mouse button**, or using the JupyterHub file tree.

After downloading it, open it up locally to visualize the run-time behavior of your application.

In [ ]:
%%bash

sbatch --partition=a100 --nodes=1 --gres=gpu:a100:1 \
    --time 00:05:00 --wait \
    --output=../output/04-work-partitioning-nsys.out --error=../output/04-work-partitioning-nsys.err \
    --wrap="nsys profile --stats=true --force-overwrite=true \
        -o ../profiles/04-work-partitioning \
        ../build/04-work-partitioning $((32 * 1024)) 256 2 8 0"

cat ../output/04-work-partitioning-nsys.out

## Next Step

The next step is parallelizing the per-patch kernel executions in notebook [05](./05-streams.ipynb).